# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Here we list all available record sets, fields, and their corresponding `@id`.
This information is necessary for targeted data extraction and analysis.

In [ ]:
# List all record sets by their `@id` and show fields with their IDs
record_sets = dataset.metadata.record_sets
print("Available record sets and their fields:")
for rs in record_sets:
    print(f'RecordSet @id: {rs.id}')
    if hasattr(rs, 'fields') and rs.fields:
        print('  Fields:')
        for field in rs.fields:
            print(f'    - {field.id} ({field.name})')
    else:
        print('  (No fields listed)')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Let's choose the main protein data record set.
# We'll use the first record set found for demonstration—update this cell with the proper `@id` if focusing on another set.

# Collect all record set ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("RecordSet IDs:", record_set_ids)

# Extract data from each record set as a pandas DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Let's preview the main protein dataset (change to the correct @id if needed)
main_rs_id = record_set_ids[0] if record_set_ids else None
if main_rs_id:
    print(f"Columns for {main_rs_id}: {dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Pick a numeric field for demonstration---for example, assume a field named 'cr:column:Abundance' exists.
if main_rs_id and ('cr:column:Abundance' in dataframes[main_rs_id].columns):
    numeric_field_id = 'cr:column:Abundance'  # Use @id, or update to an available numeric field
else:
    # Fall back to the first numeric column found
    numeric_field_id = None
    if main_rs_id:
        for col in dataframes[main_rs_id].columns:
            if pd.api.types.is_numeric_dtype(dataframes[main_rs_id][col]):
                numeric_field_id = col
                break
if not numeric_field_id:
    raise ValueError("No numeric field found for EDA.")

# Filter records: greater than a threshold
threshold = dataframes[main_rs_id][numeric_field_id].mean()  # Use mean as threshold for demo
filtered_df = dataframes[main_rs_id][dataframes[main_rs_id][numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (using mean as threshold):")
display(filtered_df.head())

# Normalize the numeric field (Z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping: try grouping by 'cr:column:Description' or another categorical field
group_field_id = None
for potential_group in ['cr:column:Description', 'cr:column:Sample', 'cr:column:Accession']:
    if main_rs_id and (potential_group in dataframes[main_rs_id].columns):
        group_field_id = potential_group
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
    display(grouped_df.head())
else:
    print("No suitable group field found for grouping analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[main_rs_id][numeric_field_id], kde=True, bins=30)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If group_field is available, show boxplot grouped by it
    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45, ha='right')
        plt.show()

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to access, explore, and conduct simple analyses of a mass spectrometry protein dataset described by a Croissant schema.

- We programmatically explored available record sets and their fields using `@id` references.
- Selected and analyzed numeric data fields, applying basic normalization and group operations.
- Produced simple visualizations for statistical inspection.

Refer to the Croissant specification for advanced features, such as merging data from multiple record sets or leveraging relationships between fields. Tailor the code to your dataset structure for domain-appropriate analyses.